Imports

In [1]:
# =========================
# REPRODUCIBILITY + IMPORTS
# =========================
from pathlib import Path
import os, random
import json, csv
import numpy as np
from collections import Counter
import logging
from typing import Optional, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
from torchvision import models
from PIL import Image

from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support

# =========================
# LOGGING SETUP
# =========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('training.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# =========================
# GLOBAL CONSTANTS
# =========================
SEED = 42

# Image preprocessing constants
CROP_MARGIN = 0.2
BORDER_ZERO_PROB = 0.7
BORDER_ZERO_MIN_FRAC = 0.03
BORDER_ZERO_MAX_FRAC = 0.10

# Augmentation constants
RANDOM_CROP_SCALE = (0.80, 1.0)
RANDOM_CROP_RATIO = (0.95, 1.05)
ROTATION_DEGREES = 4
TRANSLATE = (0.02, 0.02)
SCALE_RANGE = (0.97, 1.03)

# RGB augmentation
COLOR_JITTER_BRIGHTNESS = 0.08
COLOR_JITTER_CONTRAST = 0.12
COLOR_JITTER_SATURATION = 0.03
COLOR_JITTER_HUE = 0.01
RANDOM_ERASING_PROB = 0.25
RANDOM_ERASING_SCALE = (0.02, 0.10)
RANDOM_ERASING_RATIO = (0.3, 3.3)

# Normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

# Training constants
GRADIENT_CLIP_MAX_NORM = 1.0
VALIDATION_RESIZE_FACTOR = 1.10

# -------------------------
# Global seed / determinism
# -------------------------
def seed_everything(seed=42, deterministic=True):
    """Set random seeds for reproducibility"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    logger.info(f"Set random seed to {seed}")

seed_everything(SEED, deterministic=True)

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# =========================
# CONFIG
# =========================
class Config:
    """Training configuration"""
    # Paths
    PROCESSED_ROOT = Path("../../data/processed/Stage2/combined")
    INDEX_CSV = PROCESSED_ROOT / "index.csv"
    TRAIN_SPLIT = "train"  # "train1" / "train2" / "train3"
    TRAIN_LABELS_JSON = Path(f"../../data/labels/Stage2/{TRAIN_SPLIT}.json")
    VAL_LABELS_JSON = Path("../../data/labels/Stage2/val.json")
    TRAIN_LABELS_FALLBACK = Path("../../data/labels/Stage2/train.json")
    
    # Optional pretrained model
    PRETRAINED_MODEL_PATH = Path("../../models/classifier/Stage2/modelV10(1).pt")
    USE_PRETRAINED = False  # Set to True to load pretrained weights
    
    # Output paths
    MODEL_OUT_PATH = Path("../../models/classifier/Stage2/model.pt")
    CKPT_DIR = Path("../../models/classifier/Stage2/checkpoints")
    
    # Model architecture
    USE_RESNET = False  # Set to True to use ResNet18 instead of custom CNN
    NUM_CLASSES = 2
    IMAGE_MODE = "rgb"  # "rgb" or "gray"
    
    # Training hyperparameters
    BATCH_SIZE = 80
    LR_HEAD = 2e-3
    LR_FULL = 1e-4
    EPOCHS = 15
    IMAGE_SIZE = 256
    WEIGHT_DECAY = 4e-3
    UNFREEZE_EPOCH = 1
    
    # Scheduler settings
    SCHEDULER_PATIENCE = 3
    SCHEDULER_FACTOR = 0.5
    SCHEDULER_MIN_LR = 1e-7
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 65
    
    # Mixed precision training
    USE_AMP = True
    
    # Data loading
    NUM_WORKERS = 2
    
    @classmethod
    def create_dirs(cls):
        """Create necessary directories"""
        cls.MODEL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        cls.CKPT_DIR.mkdir(parents=True, exist_ok=True)

config = Config()
config.create_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

# =========================
# LOAD INDEX CSV
# =========================
def read_index_csv(index_csv_path: Path) -> List[Dict]:
    """Read and validate index CSV file"""
    try:
        rows = []
        with open(index_csv_path, "r", newline="") as f:
            reader = csv.DictReader(f)
            for r in reader:
                rows.append(r)
        
        if not rows:
            raise ValueError(f"index.csv is empty: {index_csv_path}")
        if "filepath" not in rows[0]:
            raise ValueError("index.csv must contain a 'filepath' column")
        
        logger.info(f"Successfully read {len(rows)} rows from index CSV")
        return rows
    
    except Exception as e:
        logger.error(f"Error reading index CSV: {e}")
        raise

index_rows = read_index_csv(config.INDEX_CSV)
logger.info(f"Example row keys: {list(index_rows[0].keys())}")
logger.info(f"Example filepath: {index_rows[0]['filepath']}")

# =========================
# LOAD LABEL MAPS
# =========================
def load_label_map(label_path: Path) -> Dict:
    """Load and validate label mappings from JSON file"""
    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    try:
        with open(label_path, "r") as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON in {label_path}: {e}")
        raise

    out = {}
    bad = []

    if isinstance(data, dict):
        for k, v in data.items():
            k_norm = k.replace("\\", "/")
            out[k_norm] = int(v)
            out[Path(k_norm).name] = int(v)
        logger.info(f"Loaded {len(out)} labels from dict format")
        return out

    if isinstance(data, list):
        for item in data:
            filepath = item.get("image", "").replace("\\", "/")
            if not filepath:
                bad.append(("MISSING_IMAGE_FIELD", item))
                continue

            fname = Path(filepath).name
            overlap = item.get("overlap", None)
            if overlap is None:
                bad.append((filepath, "overlap=None"))
                continue

            overlap = int(overlap)
            if overlap == 1:
                class_id = 0
            elif overlap == 0:
                class_id = 1
            else:
                bad.append((filepath, f"overlap={overlap}"))
                continue

            out[filepath] = class_id
            out[fname] = class_id

        if bad:
            logger.warning(f"Found {len(bad)} inconsistent/bad label rows (showing up to 10):")
            for row in bad[:10]:
                logger.warning(f"  {row}")
            raise ValueError(f"Inconsistent labels for {len(bad)} samples in {label_path}")

        logger.info(f"Loaded {len(out)} labels from list format")
        return out

    raise ValueError("Label JSON must be a dict or list")

# Load labels with fallback
try:
    if config.TRAIN_LABELS_JSON.exists():
        train_label_map = load_label_map(config.TRAIN_LABELS_JSON)
        logger.info(f"Loaded train labels from: {config.TRAIN_LABELS_JSON}")
    else:
        train_label_map = load_label_map(config.TRAIN_LABELS_FALLBACK)
        logger.warning(f"Train label not found, using fallback: {config.TRAIN_LABELS_FALLBACK}")
    
    val_label_map = load_label_map(config.VAL_LABELS_JSON)
    
    logger.info(f"Train label distribution: {Counter(train_label_map.values())}")
    logger.info(f"Val label distribution: {Counter(val_label_map.values())}")

except Exception as e:
    logger.error(f"Failed to load labels: {e}")
    raise

# =========================
# CROPS / AUGS
# =========================
def crop_to_tray_interior(img: Image.Image) -> Image.Image:
    """Crop image to interior region, removing borders"""
    w, h = img.size
    return img.crop((
        int(w * CROP_MARGIN),
        int(h * CROP_MARGIN),
        int(w * (1 - CROP_MARGIN)),
        int(h * (1 - CROP_MARGIN)),
    ))

class RandomBorderZero:
    """Custom augmentation to zero out image borders"""
    def __init__(self, p=BORDER_ZERO_PROB, min_frac=BORDER_ZERO_MIN_FRAC, max_frac=BORDER_ZERO_MAX_FRAC):
        self.p = p
        self.min_frac = min_frac
        self.max_frac = max_frac

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if torch.rand(1).item() > self.p:
            return x
        _, h, w = x.shape
        frac = float(torch.empty(1).uniform_(self.min_frac, self.max_frac))
        bx = int(w * frac)
        by = int(h * frac)
        x = x.clone()
        x[:, :by, :] = 0
        x[:, h-by:, :] = 0
        x[:, :, :bx] = 0
        x[:, :, w-bx:] = 0
        return x

# =========================
# DATASET
# =========================
_MISS = object()

class ProcessedSplitDataset(Dataset):
    """Dataset for loading processed tray images with labels"""
    def __init__(
        self,
        index_rows: List[Dict],
        processed_root: Path,
        split: str,
        label_map: Dict,
        transform=None,
        auto_label_stage1_missing: bool = False,
        stage1_default_class: int = 1,
        verify_files_exist: bool = True,
    ):
        self.processed_root = processed_root
        self.split = split.lower().strip()
        self.transform = transform
        self.label_map = label_map

        # Select rows for this split
        rows = []
        for r in index_rows:
            fp = (r.get("filepath") or "").replace("\\", "/").strip()
            if not fp:
                continue

            row_split = (r.get("split") or "").strip().lower()
            if row_split:
                if row_split == self.split:
                    rows.append(r)
            else:
                if f"images/{self.split}/" in fp:
                    rows.append(r)

        if not rows:
            raise ValueError(f"No samples found for split='{self.split}'")

        self.filepaths = []
        self.labels = []

        missing_hard = []
        missing_files = []
        corrupt_files = []
        auto_filled = 0

        for r in rows:
            fp_norm = (r.get("filepath") or "").replace("\\", "/").strip()
            fname = Path(fp_norm).name
            stage = (r.get("stage") or "").strip().lower()

            # Handle train1/train2/train3 splits
            label_split = "train" if self.split in {"train1", "train2", "train3"} else self.split

            keys_to_try = [
                fp_norm,
                f"images/{label_split}/{fname}",
                fname,
            ]

            label = _MISS
            for k in keys_to_try:
                label = self.label_map.get(k, _MISS)
                if label is not _MISS:
                    break

            if label is _MISS:
                if auto_label_stage1_missing and stage == "stage1":
                    label = stage1_default_class
                    auto_filled += 1
                else:
                    missing_hard.append((fp_norm, stage))
                    continue

            if verify_files_exist:
                img_path = self.processed_root / fp_norm
                if not img_path.exists():
                    missing_files.append(str(img_path))
                    continue
                
                # Verify image integrity
                try:
                    with Image.open(img_path) as img:
                        img.verify()
                except Exception:
                    corrupt_files.append(str(img_path))
                    continue

            self.filepaths.append(fp_norm)
            self.labels.append(int(label))

        if auto_filled > 0:
            logger.info(f"Auto-filled {auto_filled} missing labels as class={stage1_default_class}")

        if corrupt_files:
            logger.error(f"Found {len(corrupt_files)} corrupt image files (showing first 10):")
            for p in corrupt_files[:10]:
                logger.error(f"  {p}")
            raise ValueError(f"Corrupt files detected: {len(corrupt_files)} samples")

        if missing_files:
            logger.error(f"Missing image files for {len(missing_files)} samples (showing first 20):")
            for p in missing_files[:20]:
                logger.error(f"  {p}")
            raise FileNotFoundError(f"Missing files for {len(missing_files)} samples")

        if missing_hard:
            logger.error(f"Missing labels for {len(missing_hard)} samples (showing first 30):")
            for fp_norm, stage in missing_hard[:30]:
                logger.error(f"  {fp_norm} | stage: {stage}")
            raise KeyError(f"Missing labels for {len(missing_hard)} samples")

        logger.info(f"Dataset '{split}': {len(self.filepaths)} samples loaded")

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        rel_path = self.filepaths[idx]
        img_path = self.processed_root / rel_path

        try:
            img = Image.open(img_path)
            img = img.convert("L" if config.IMAGE_MODE == "gray" else "RGB")
            img = crop_to_tray_interior(img)

            if self.transform:
                img = self.transform(img)

            y = torch.tensor(self.labels[idx], dtype=torch.long)
            return img, y
        
        except Exception as e:
            logger.error(f"Error loading image {img_path}: {e}")
            raise

# =========================
# TRANSFORMS
# =========================
in_channels = 1 if config.IMAGE_MODE == "gray" else 3

if config.IMAGE_MODE == "gray":
    train_transform = T.Compose([
        T.RandomResizedCrop(config.IMAGE_SIZE, scale=RANDOM_CROP_SCALE, ratio=RANDOM_CROP_RATIO),
        T.RandomAffine(degrees=ROTATION_DEGREES, translate=TRANSLATE, scale=SCALE_RANGE),
        T.ToTensor(),
        T.Normalize(GRAY_MEAN, GRAY_STD),
        RandomBorderZero(),
    ])

    val_transform = T.Compose([
        T.Resize(int(config.IMAGE_SIZE * VALIDATION_RESIZE_FACTOR)),
        T.CenterCrop(config.IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(GRAY_MEAN, GRAY_STD),
    ])
else:
    train_transform = T.Compose([
        T.RandomResizedCrop(config.IMAGE_SIZE, scale=RANDOM_CROP_SCALE, ratio=RANDOM_CROP_RATIO),
        T.RandomAffine(degrees=ROTATION_DEGREES, translate=TRANSLATE, scale=SCALE_RANGE),
        T.ColorJitter(
            brightness=COLOR_JITTER_BRIGHTNESS,
            contrast=COLOR_JITTER_CONTRAST,
            saturation=COLOR_JITTER_SATURATION,
            hue=COLOR_JITTER_HUE
        ),
        T.ToTensor(),
        T.RandomErasing(p=RANDOM_ERASING_PROB, scale=RANDOM_ERASING_SCALE, ratio=RANDOM_ERASING_RATIO),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        RandomBorderZero(),
    ])
    
    val_transform = T.Compose([
        T.Resize(int(config.IMAGE_SIZE * VALIDATION_RESIZE_FACTOR)),
        T.CenterCrop(config.IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

# =========================
# CREATE DATASETS
# =========================
try:
    train_ds = ProcessedSplitDataset(
        index_rows, config.PROCESSED_ROOT, config.TRAIN_SPLIT, train_label_map,
        transform=train_transform,
        auto_label_stage1_missing=False,
        verify_files_exist=True
    )

    val_ds = ProcessedSplitDataset(
        index_rows, config.PROCESSED_ROOT, "val", val_label_map,
        transform=val_transform,
        auto_label_stage1_missing=False,
        verify_files_exist=True
    )

    logger.info(f"Train split: {config.TRAIN_SPLIT}")
    logger.info(f"Train distribution: {Counter(train_ds.labels)}")
    logger.info(f"Val distribution: {Counter(val_ds.labels)}")

except Exception as e:
    logger.error(f"Failed to create datasets: {e}")
    raise

# =========================
# CLASS WEIGHTS
# =========================
counts = np.bincount(train_ds.labels, minlength=config.NUM_CLASSES)
counts = np.maximum(counts, 1)
class_weights = torch.tensor(
    [1.0 / counts[i] for i in range(config.NUM_CLASSES)],
    dtype=torch.float
).to(device)

logger.info(f"Class counts: {counts}")
logger.info(f"Class weights: {class_weights.tolist()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# =========================
# LOADERS
# =========================
train_loader = DataLoader(
    train_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=g,
    pin_memory=True,
)

# Verify data loading
try:
    x, y = next(iter(train_loader))
    logger.info(f"Batch image shape: {x.shape} | Batch label shape: {y.shape}")
except Exception as e:
    logger.error(f"Error in data loading: {e}")
    raise

# =========================
# MODEL
# =========================
class SimpleCNN_GAP(nn.Module):
    """Custom CNN with Global Average Pooling"""
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


def create_model(use_resnet=False, in_channels=3, num_classes=2, pretrained=True):
    """Create model - either ResNet or custom CNN"""
    if use_resnet:
        logger.info("Creating ResNet18 model")
        model = models.resnet18(pretrained=pretrained)
        
        # Modify first conv layer if using grayscale
        if in_channels == 1:
            model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Replace final layer
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    else:
        logger.info("Creating custom SimpleCNN_GAP model")
        return SimpleCNN_GAP(in_channels=in_channels, num_classes=num_classes)


# Create model
model = create_model(
    use_resnet=config.USE_RESNET,
    in_channels=in_channels,
    num_classes=config.NUM_CLASSES,
    pretrained=True
).to(device)

# Load pretrained weights if specified
if config.USE_PRETRAINED and config.PRETRAINED_MODEL_PATH.exists():
    try:
        state_dict = torch.load(config.PRETRAINED_MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict, strict=True)
        logger.info(f"Loaded pretrained weights from: {config.PRETRAINED_MODEL_PATH}")
    except Exception as e:
        logger.warning(f"Could not load pretrained weights: {e}")
        logger.info("Continuing with randomly initialized weights")
elif config.USE_PRETRAINED:
    logger.warning(f"Pretrained weights not found at {config.PRETRAINED_MODEL_PATH}")
    logger.info("Starting with randomly initialized weights")


def count_parameters(m):
    """Count trainable parameters in model"""
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


total_params = count_parameters(model)
logger.info(f"Total trainable parameters: {total_params:,}")

# Freeze feature extractor initially
if config.USE_RESNET:
    # For ResNet, freeze all except final layer
    for name, param in model.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False
else:
    # For custom CNN, freeze features
    for p in model.features.parameters():
        p.requires_grad = False

trainable_params = count_parameters(model)
logger.info(f"Trainable parameters (head only): {trainable_params:,}")

# =========================
# OPTIMIZER & SCHEDULER
# =========================
optimizer = optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=config.LR_HEAD,
    weight_decay=config.WEIGHT_DECAY,
)

# Initialize scheduler as None (will be created after unfreezing)
scheduler = None

# Mixed precision scaler
scaler = GradScaler() if config.USE_AMP else None

# =========================
# METRICS TRACKING
# =========================
class MetricsTracker:
    """Track and log training metrics"""
    def __init__(self):
        self.train_losses = []
        self.train_accs = []
        self.val_losses = []
        self.val_accs = []
        self.learning_rates = []
        self.best_val_loss = float('inf')
        self.best_epoch = 0
    
    def update(self, epoch, train_loss, train_acc, val_loss, val_acc, lr):
        self.train_losses.append(train_loss)
        self.train_accs.append(train_acc)
        self.val_losses.append(val_loss)
        self.val_accs.append(val_acc)
        self.learning_rates.append(lr)
        
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_epoch = epoch
    
    def save(self, path):
        """Save metrics to file"""
        metrics = {
            'train_losses': self.train_losses,
            'train_accs': self.train_accs,
            'val_losses': self.val_losses,
            'val_accs': self.val_accs,
            'learning_rates': self.learning_rates,
            'best_val_loss': self.best_val_loss,
            'best_epoch': self.best_epoch,
        }
        with open(path, 'w') as f:
            json.dump(metrics, f, indent=2)
        logger.info(f"Saved metrics to {path}")

metrics_tracker = MetricsTracker()

# =========================
# TRAINING FUNCTIONS
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None, use_amp=False):
    """Train for one epoch"""
    model.train()
    train_loss_sum, train_correct, train_total = 0.0, 0, 0
    
    try:
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            
            if use_amp and scaler is not None:
                with autocast():
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_MAX_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(imgs)
                loss = criterion(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_MAX_NORM)
                optimizer.step()

            train_loss_sum += loss.item() * imgs.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_sum / max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)
        
        return train_loss, train_acc
    
    except RuntimeError as e:
        if "out of memory" in str(e):
            logger.error("CUDA out of memory! Try reducing batch size.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        raise


@torch.no_grad()
def validate(model, loader, criterion, device, num_classes=2):
    """Validate model"""
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    preds_all, y_all, probs_all = [], [], []

    try:
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(imgs)
            loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            val_loss_sum += loss.item() * imgs.size(0)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            preds_all.extend(preds.cpu().tolist())
            y_all.extend(labels.cpu().tolist())
            probs_all.extend(probs.cpu().numpy())

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)
        
        # Calculate additional metrics
        probs_array = np.array(probs_all)
        
        # Precision, Recall, F1
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_all, preds_all, average='weighted', zero_division=0
        )
        
        # ROC-AUC (handle binary case)
        try:
            if num_classes == 2:
                roc_auc = roc_auc_score(y_all, probs_array[:, 1])
            else:
                roc_auc = roc_auc_score(y_all, probs_array, multi_class='ovr', average='weighted')
        except ValueError:
            roc_auc = 0.0
        
        # Confusion matrix
        cm = np.zeros((num_classes, num_classes), dtype=int)
        for p, t in zip(preds_all, y_all):
            cm[t, p] += 1
        
        metrics = {
            'loss': val_loss,
            'accuracy': val_acc,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'predictions': preds_all,
            'labels': y_all,
            'probabilities': probs_array,
            'confusion_matrix': cm,
        }
        
        return metrics
    
    except Exception as e:
        logger.error(f"Error during validation: {e}")
        raise


# =========================
# TRAIN LOOP
# =========================
logger.info("=" * 80)
logger.info("Starting training")
logger.info("=" * 80)

epochs_without_improvement = 0

for epoch in range(1, config.EPOCHS + 1):
    
    # Unfreeze at specified epoch
    if epoch == config.UNFREEZE_EPOCH:
        logger.info(f"\n{'='*80}")
        logger.info(f"Unfreezing features at epoch {epoch}")
        logger.info(f"{'='*80}\n")
        
        # Unfreeze all parameters
        for p in model.parameters():
            p.requires_grad = True
        
        # Create new optimizer with lower learning rate
        optimizer = optim.Adam(
            model.parameters(),
            lr=config.LR_FULL,
            weight_decay=config.WEIGHT_DECAY,
        )
        
        # Create scheduler
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            patience=config.SCHEDULER_PATIENCE,
            factor=config.SCHEDULER_FACTOR,
            min_lr=config.SCHEDULER_MIN_LR,
            # verbose=True  # Removed - not supported in newer PyTorch versions
        )
        
        trainable_params = count_parameters(model)
        logger.info(f"Trainable parameters (full model): {trainable_params:,}")

    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device,
        scaler=scaler, use_amp=config.USE_AMP
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, device, config.NUM_CLASSES)
    val_loss = val_metrics['loss']
    val_acc = val_metrics['accuracy']

    # Step scheduler if it exists
    if scheduler is not None:
        scheduler.step(val_loss)

    # Get current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    
    # Update metrics tracker
    metrics_tracker.update(epoch, train_loss, train_acc, val_loss, val_acc, current_lr)

    # Save checkpoint
    checkpoint_dict = {
        "epoch": epoch,
        "train_split": config.TRAIN_SPLIT,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_metrics": val_metrics,
        "seed": SEED,
        "config": vars(config),
    }
    
    if scheduler is not None:
        checkpoint_dict["scheduler_state"] = scheduler.state_dict()
    
    if scaler is not None:
        checkpoint_dict["scaler_state"] = scaler.state_dict()
    
    ckpt_path = config.CKPT_DIR / f"{config.TRAIN_SPLIT}_epoch_{epoch:03d}.pt"
    torch.save(checkpoint_dict, ckpt_path)
    logger.info(f"Saved checkpoint: {ckpt_path}")

    # Save best model
    if val_loss < metrics_tracker.best_val_loss:
        epochs_without_improvement = 0
        
        # Save just the model weights (for inference)
        best_path = config.CKPT_DIR / f"{config.TRAIN_SPLIT}_best.pt"
        torch.save(model.state_dict(), best_path)
        
        # Save complete checkpoint (for resuming training)
        best_checkpoint_path = config.CKPT_DIR / f"{config.TRAIN_SPLIT}_best_checkpoint.pt"
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state": scaler.state_dict() if scaler is not None else None,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_metrics": val_metrics,
            "train_split": config.TRAIN_SPLIT,
            "seed": SEED,
            "config": vars(config),
        }, best_checkpoint_path)
        
        logger.info(f"✓ New best model saved: {best_path} (val_loss={val_loss:.4f})")
        logger.info(f"✓ Best checkpoint saved: {best_checkpoint_path}")
    else:
        epochs_without_improvement += 1

    # Log epoch results
    logger.info(f"\n{'='*80}")
    logger.info(f"Epoch [{epoch}/{config.EPOCHS}]")
    logger.info(f"{'='*80}")
    logger.info(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.3f}")
    logger.info(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.3f}")
    logger.info(f"Precision:  {val_metrics['precision']:.3f} | Recall: {val_metrics['recall']:.3f}")
    logger.info(f"F1 Score:   {val_metrics['f1']:.3f} | ROC-AUC: {val_metrics['roc_auc']:.3f}")
    logger.info(f"Learning Rate: {current_lr:.6f}")
    logger.info(f"Val predictions: {Counter(val_metrics['predictions'])}")
    logger.info(f"Val true labels: {Counter(val_metrics['labels'])}")
    logger.info(f"Confusion Matrix (rows=true, cols=pred):\n{val_metrics['confusion_matrix']}")
    
    # Check for model collapse
    if len(set(val_metrics['predictions'])) == 1:
        logger.warning("⚠ COLLAPSE DETECTED: model predicts only one class on validation")
    
    # Early stopping check
    if epochs_without_improvement >= config.EARLY_STOPPING_PATIENCE:
        logger.info(f"\n{'='*80}")
        logger.info(f"Early stopping triggered after {epoch} epochs")
        logger.info(f"Best validation loss: {metrics_tracker.best_val_loss:.4f} at epoch {metrics_tracker.best_epoch}")
        logger.info(f"{'='*80}\n")
        break

# =========================
# SAVE FINAL MODEL & METRICS
# =========================
torch.save(model.state_dict(), config.MODEL_OUT_PATH)
logger.info(f"Saved final model to: {config.MODEL_OUT_PATH}")

metrics_path = config.MODEL_OUT_PATH.parent / "training_metrics.json"
metrics_tracker.save(metrics_path)

logger.info(f"\n{'='*80}")
logger.info("Training completed!")
logger.info(f"Best validation loss: {metrics_tracker.best_val_loss:.4f} at epoch {metrics_tracker.best_epoch}")
logger.info(f"{'='*80}\n")

2026-02-02 13:04:19,704 - INFO - Set random seed to 42
2026-02-02 13:04:19,731 - INFO - Using device: cuda
2026-02-02 13:04:19,749 - INFO - GPU: NVIDIA GeForce RTX 4060 Laptop GPU
2026-02-02 13:04:19,750 - INFO - CUDA Version: 12.8
2026-02-02 13:04:19,756 - INFO - Successfully read 1975 rows from index CSV
2026-02-02 13:04:19,756 - INFO - Example row keys: ['filepath', 'split', 'stage', 'scan_session_id', 'filename', 'scan_timestamp', 'source_interim_path', 'group_id']
2026-02-02 13:04:19,757 - INFO - Example filepath: images/train/stage1__2026-01-21_10-37-50-052_aug00.png
2026-02-02 13:04:19,762 - INFO - Loaded 3350 labels from list format
2026-02-02 13:04:19,763 - INFO - Loaded train labels from: ../../data/labels/Stage2/train.json
2026-02-02 13:04:19,764 - INFO - Loaded 390 labels from list format
2026-02-02 13:04:19,764 - INFO - Train label distribution: Counter({0: 1870, 1: 1480})
2026-02-02 13:04:19,764 - INFO - Val label distribution: Counter({0: 220, 1: 170})
2026-02-02 13:04:2